In [1]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

ModuleNotFoundError: No module named 'langgraph'

In [4]:
load_dotenv()
model = ChatGroq(model="llama-3.3-70b-versatile")

In [5]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [6]:
def chat_node(state: ChatState):
    query = state["messages"]
    response = model.invoke(query)

    return {"messages": [response]}

In [7]:
checkpointer = MemorySaver()

graph = StateGraph(ChatState)

graph.add_node("chat_node", chat_node)

graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

chatbot = graph.compile(checkpointer=checkpointer)

In [8]:
thread_id = "1"

while True:
    user_input = input("Type here: ")

    if user_input.strip().lower() in ["exit", "quit", "q"]:
        break

    config = {"configurable": {"thread_id": thread_id}}

    response = chatbot.invoke(
        {"messages": [HumanMessage(content=user_input.strip())]}, config=config
    )

    print(f"AI: {response['messages'][-1].content}")